In [1]:
# ============================================================

# CELL 1 — Install / Import Libraries

# ============================================================

# If necessary, uncomment the following line:

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import re
import warnings

warnings.filterwarnings("ignore")

# Download NLTK resources

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("Libraries imported successfully.")

# ============================================================

# CELL 2 — Load Dataset

# ============================================================

file_path = "Cleaned_DS_jobs2.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

# ============================================================

# CELL 3 — Inspect Dataset

# ============================================================

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

# ============================================================

# CELL 4 — Remove Duplicate Rows

# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

# ============================================================

# CELL 5 — Identify Text Columns

# ============================================================

# Automatically find columns containing text data

text_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Text columns found:")
for col in text_columns:
print("-", col)

# ============================================================

# CELL 6 — Combine Text Columns

# ============================================================

# NLP works best when the relevant text information is combined

# into one document per job.

if len(text_columns) == 0:
raise ValueError("No text columns were found in the dataset.")

df["combined_text"] = (
df[text_columns]
.fillna("")
.astype(str)
.agg(" ".join, axis=1)
)

print("Combined text created.")
display(df[["combined_text"]].head())

# ============================================================

# CELL 7 — NLP Text Cleaning

# ============================================================

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
"""
Clean job-related text:
- Convert to lowercase
- Remove URLs
- Remove special characters
- Remove numbers
- Remove stopwords
- Lemmatize words
"""

```
text = str(text).lower()

# Remove URLs
text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

# Keep alphabetic characters
text = re.sub(r"[^a-zA-Z\s]", " ", text)

# Remove extra spaces
text = re.sub(r"\s+", " ", text).strip()

words = text.split()

cleaned_words = [
    lemmatizer.lemmatize(word)
    for word in words
    if word not in stop_words and len(word) > 2
]

return " ".join(cleaned_words)
```

df["cleaned_text"] = df["combined_text"].apply(clean_text)

print("NLP cleaning completed.")

display(df[["combined_text", "cleaned_text"]].head())

# ============================================================

# CELL 8 — TF-IDF Vectorization

# ============================================================

# TF-IDF converts job descriptions/titles into numerical

# features that K-Means can use.

vectorizer = TfidfVectorizer(
max_features=5000,
min_df=2,
max_df=0.95,
ngram_range=(1, 2),
sublinear_tf=True
)

X = vectorizer.fit_transform(df["cleaned_text"])

print("TF-IDF matrix created.")
print("Number of documents:", X.shape[0])
print("Number of features:", X.shape[1])

# ============================================================

# CELL 9 — Find Optimal Number of Clusters

# Using the Elbow Method

# ============================================================

inertia = []

# Test different values of K

K_range = range(2, 11)

for k in K_range:
kmeans = KMeans(
n_clusters=k,
random_state=42,
n_init=10
)

```
kmeans.fit(X)
inertia.append(kmeans.inertia_)
```

plt.figure(figsize=(10, 6))

plt.plot(
list(K_range),
inertia,
marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (Inertia)")
plt.title("Elbow Method for Selecting K")
plt.xticks(list(K_range))
plt.grid(True)

plt.show()

# ============================================================

# CELL 10 — Silhouette Score

# ============================================================

silhouette_scores = {}

for k in K_range:

```
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(X)

score = silhouette_score(X, labels)

silhouette_scores[k] = score

print(f"K = {k}, Silhouette Score = {score:.4f}")
```

# Plot silhouette scores

plt.figure(figsize=(10, 6))

plt.plot(
list(silhouette_scores.keys()),
list(silhouette_scores.values()),
marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")
plt.grid(True)

plt.show()

# ============================================================

# CELL 11 — Select Best K Automatically

# ============================================================

best_k = max(
silhouette_scores,
key=silhouette_scores.get
)

print("Best K according to Silhouette Score:", best_k)

# ============================================================

# CELL 12 — Train Final K-Means Model

# ============================================================

kmeans = KMeans(
n_clusters=best_k,
random_state=42,
n_init=20
)

df["cluster"] = kmeans.fit_predict(X)

print("K-Means clustering completed.")

# ============================================================

# CELL 13 — Number of Jobs in Each Cluster

# ============================================================

cluster_counts = (
df["cluster"]
.value_counts()
.sort_index()
)

print("Number of jobs in each cluster:")

display(
cluster_counts.to_frame("Number_of_Jobs")
)

# ============================================================

# CELL 14 — Visualize Clusters Using PCA-like SVD

# ============================================================

# TF-IDF can have thousands of dimensions.

# SVD reduces it to two dimensions for visualization.

svd = TruncatedSVD(
n_components=2,
random_state=42
)

X_2d = svd.fit_transform(X)

plot_df = pd.DataFrame({
"Component_1": X_2d[:, 0],
"Component_2": X_2d[:, 1],
"Cluster": df["cluster"]
})

plt.figure(figsize=(12, 8))

sns.scatterplot(
data=plot_df,
x="Component_1",
y="Component_2",
hue="Cluster",
palette="tab10",
s=70,
alpha=0.7
)

plt.title("K-Means Clusters of Data Science Jobs")
plt.xlabel("SVD Component 1")
plt.ylabel("SVD Component 2")
plt.legend(title="Cluster")

plt.show()

# ============================================================

# CELL 15 — Find Important Words in Each Cluster

# ============================================================

terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

cluster_keywords = {}

for cluster_number in range(best_k):

```
top_words = [
    terms[index]
    for index in order_centroids[cluster_number, :15]
]

cluster_keywords[cluster_number] = top_words

print(
    f"\nCluster {cluster_number}:"
)

print(", ".join(top_words))
```

# ============================================================

# CELL 16 — Create Cluster Summary

# ============================================================

cluster_summary = []

for cluster_number in range(best_k):

```
cluster_data = df[df["cluster"] == cluster_number]

cluster_summary.append({
    "Cluster": cluster_number,
    "Number_of_Jobs": len(cluster_data),
    "Top_Keywords": ", ".join(
        cluster_keywords[cluster_number]
    )
})
```

cluster_summary_df = pd.DataFrame(cluster_summary)

display(cluster_summary_df)

# ============================================================

# CELL 17 — Display Example Jobs from Each Cluster

# ============================================================

for cluster_number in range(best_k):

```
print("\n" + "=" * 70)
print(f"CLUSTER {cluster_number}")
print("=" * 70)

cluster_data = df[
    df["cluster"] == cluster_number
]

display(
    cluster_data.head(5)
)
```

# ============================================================

# CELL 18 — Inspect Job Titles by Cluster

# ============================================================

# Try to automatically find a likely job-title column

possible_title_columns = [
col for col in df.columns
if any(
keyword in col.lower()
for keyword in [
"job title",
"job_title",
"title",
"position",
"role"
]
)
]

print("Possible job title columns:")
print(possible_title_columns)

if possible_title_columns:

```
title_column = possible_title_columns[0]

for cluster_number in range(best_k):
    
    print(
        f"\nCluster {cluster_number} — Job Titles"
    )
    
    titles = (
        df[df["cluster"] == cluster_number][title_column]
        .value_counts()
        .head(10)
    )
    
    display(titles.to_frame("Frequency"))
```

else:
print("No obvious job-title column was detected.")

# ============================================================

# CELL 19 — Save Results

# ============================================================

output_file = "DS_jobs_kmeans_clusters.csv"

df.to_csv(
output_file,
index=False
)

cluster_summary_df.to_csv(
"DS_jobs_cluster_summary.csv",
index=False
)

print("Files saved:")
print("-", output_file)
print("- DS_jobs_cluster_summary.csv")

# ============================================================

# CELL 20 — Final Results

# ============================================================

print("=" * 70)
print("K-MEANS NLP ANALYSIS COMPLETE")
print("=" * 70)

print(f"Original dataset rows: {len(df)}")
print(f"Number of clusters: {best_k}")
print(f"TF-IDF features: {X.shape[1]}")

print("\nCluster sizes:")
display(cluster_counts.to_frame("Number_of_Jobs"))

print("\nCluster keywords:")
display(cluster_summary_df)


IndentationError: expected an indented block after 'for' statement on line 91 (2517545643.py, line 92)